## Downloading all of the images
Creating a gallery-like effect using this resource and manually uploading all the artwork images: https://spirale-img.vercel.app/?mcp_token=eyJwaWQiOjM0NjIwMjgsInNpZCI6MTc1Nzk4ODY1NCwiYXgiOiI5NjA3NDEzZDNjZjI2MTZiYTBjNjdjZjYwZWQ3Yjg2YSIsInRzIjoxNzc3NDgxMzA3LCJleHAiOjE3Nzk5MDA1MDd9.T7mFtuyZGjDa5mkASx-wtbrSxixb1sFSU73sUkq2jx0

In [1]:
import pandas as pd
import requests
import os
import shutil
import time
from pathlib import Path

In [3]:
def save_img(url, fpath):
    if 'ids.si.edu' in url:
        url = url + '&max=400'
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
    response.raise_for_status()
    with open(fpath, 'wb') as f:
        f.write(response.content)

In [4]:
day = "04-26"

In [5]:
df = pd.read_csv('../../data/downloaded/processed/' + day + '/artworks.csv')
print(f"Loaded {len(df)} artworks ({df['image_url'].isna().sum()} without image URLs)")

Loaded 405 artworks (65 without image URLs)


In [10]:
os.makedirs('images', exist_ok=True)
failed = []

for _, row in df.iterrows():
    url = row['image_url']
    if pd.isna(url):
        continue
    try:
        save_img(str(url), f"images/{row['id']}.jpg")
    except Exception as e:
        print(f"Failed {row['id']}: {e}")
        failed.append(row['id'])

downloaded = len(list(Path('images').glob('*.jpg')))

if failed:
    print("Failed IDs:", failed)

Failed W127: 403 Client Error: Forbidden for url: https://www.artic.edu/iiif/2/8397dcc0-fd97-1bf1-dc42-c207372daf00/full/843,/0/default.jpg
Failed W165: 404 Client Error: Not Found for url: https://www.clevelandart.org/_next/image?url=https%E2%80%A6Cd6152%5Cu243061522%5C2015.145_o2.jpg&w=3840&q=75
Failed IDs: ['W127', 'W165']


### Manual fixes
Manually downloading for W127: https://www.artic.edu/iiif/2/8397dcc0-fd97-1bf1-dc42-c207372daf00/full/843,/0/default.jpg

For the CMA link - that image is not covered under CC0 so not downloading


Need to shuffle the order for the tool I'm uploading the images to

In [11]:
SEED = 42
IMG_EXT = '.jpg'

downloaded_ids = {f.stem for f in Path('images').glob(f'*{IMG_EXT}')}
df_downloaded = df[df['id'].astype(str).isin(downloaded_ids)].copy()

shuffled = df_downloaded.sample(frac=1, random_state=SEED).reset_index(drop=True)
shuffled['new_name'] = (shuffled.index + 1).astype(str)

print(f"{len(shuffled)} files will be renamed")
print(shuffled[['artist', 'id', 'new_name']].head(10))

338 files will be renamed
   artist    id new_name
0       0  W116        1
1      21  W404        2
2      21  W227        3
3       0  W079        4
4      22  W253        5
5      21  W241        6
6       5  W043        7
7      25  W303        8
8      22  W251        9
9       5  W026       10


In [12]:
TEMP_DIR = 'images/_rename_tmp'
os.makedirs(TEMP_DIR, exist_ok=True)

for _, row in shuffled.iterrows():
    shutil.move(f"images/{row['id']}{IMG_EXT}", os.path.join(TEMP_DIR, row['new_name'] + IMG_EXT))

for f in os.listdir(TEMP_DIR):
    shutil.move(os.path.join(TEMP_DIR, f), f"images/{f}")

os.rmdir(TEMP_DIR)